# Summarize Evaluation Results

This notebook loads completed evaluation results, aggregates scores by benchmark/category/supercategory, and generates Markdown and HTML reports.

## Workflow

1. Load result JSONs from `results/<model-name>/` directories
2. Build comparison table across models
3. Aggregate by category / supercategory (KMMLU, CLIcK)
4. Export to Markdown (`results/RESULTS.md`)
5. Generate standalone HTML report with charts

> **Prerequisite**: Run the "Save Results to JSON" cell in `1_kmcq_benchmark.ipynb` first to populate `results/<model>/`.

## Step 1: Configuration

In [28]:
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
RESULTS_DIR = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Exists: {RESULTS_DIR.exists()}")

Project root: /Users/hyochoi/dev/rhoai-evalhub-lab
Results dir:  /Users/hyochoi/dev/rhoai-evalhub-lab/results
Exists: True


## Step 2: Load Results

Results are organized as `results/<model-name>/<job-name>.json`.
Each JSON file is in EvalHub format:

```json
{
  "model": {"name": "qwen3-14b", "url": "..."},
  "benchmarks": [{"id": "kmmlu", "metrics": {"overall_accuracy": 60}}]
}
```

In [29]:
def load_results(results_dir: Path) -> dict:
    """Load all result JSONs organized by model.
    Returns {model_name: {job_file_stem: data_dict}}.
    """
    all_results = {}
    if not results_dir.exists():
        print(f"Results directory not found: {results_dir}")
        return all_results

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        all_results[model_name] = {}
        for json_file in sorted(model_dir.glob("*.json")):
            try:
                with open(json_file) as f:
                    data = json.load(f)
                all_results[model_name][json_file.stem] = data
            except (json.JSONDecodeError, IOError) as e:
                print(f"  Skipped {json_file}: {e}")
    return all_results


all_results = load_results(RESULTS_DIR)
print(f"Loaded results for {len(all_results)} model(s):")
for model, jobs in all_results.items():
    print(f"  {model}: {len(jobs)} job(s)")

Loaded results for 5 model(s):
  exaone4-32b: 27 job(s)
  gemma4-12b: 10 job(s)
  glm-53-flash: 17 job(s)
  qwen3-14b: 43 job(s)
  qwen36-27b: 6 job(s)


## Step 3: Build Comparison Table

Extract `overall_accuracy` from each benchmark and build a `{task: {model: score}}` table.

In [30]:
def extract_benchmark_scores(data: dict) -> dict:
    """Extract {benchmark_id: overall_accuracy} from a single result file."""
    scores = {}
    for bm in data.get("benchmarks", []):
        bm_id = bm.get("id", "unknown")
        metrics = bm.get("metrics", {})
        acc = metrics.get("overall_accuracy")
        if acc is not None:
            scores[bm_id] = round(float(acc), 2)
    return scores


def build_score_table(all_results: dict) -> dict:
    """Build {benchmark: {model: best_score}}.
    If a model has multiple jobs for the same benchmark, keep the latest (last file).
    """
    table = {}
    for model_name, jobs in all_results.items():
        for job_file, data in jobs.items():
            for bm_id, score in extract_benchmark_scores(data).items():
                if bm_id not in table:
                    table[bm_id] = {}
                table[bm_id][model_name] = score
    return table


score_table = build_score_table(all_results)
print(f"Benchmarks found: {sorted(score_table.keys())}")
for bm, models in sorted(score_table.items()):
    scores_str = ", ".join(f"{m}={s}%" for m, s in sorted(models.items()))
    print(f"  {bm}: {scores_str}")

Benchmarks found: ['click', 'haerae', 'kmmlu', 'kmmlu_hard', 'kobest_boolq']
  click: exaone4-32b=68.3%, gemma4-12b=73.88%, glm-53-flash=94.96%, qwen3-14b=66.82%, qwen36-27b=75.9%
  haerae: exaone4-32b=63.2%, gemma4-12b=69.87%, glm-53-flash=77.01%, qwen3-14b=57.14%, qwen36-27b=16.0%
  kmmlu: exaone4-32b=52.24%, gemma4-12b=57.51%, glm-53-flash=86.48%, qwen3-14b=53.37%, qwen36-27b=62.5%
  kmmlu_hard: exaone4-32b=29.48%, gemma4-12b=33.8%, glm-53-flash=79.21%, qwen3-14b=27.51%, qwen36-27b=43.06%
  kobest_boolq: exaone4-32b=91.52%, gemma4-12b=96.08%, glm-53-flash=97.77%, qwen3-14b=93.23%, qwen36-27b=96.65%


## Step 4: Display Results Table

In [31]:
try:
    import pandas as pd

    if score_table:
        df = pd.DataFrame(score_table).T
        df.index.name = "Benchmark"
        df = df.sort_index()
        display(df.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
    else:
        print("No results to display. Run evaluations first.")
except ImportError:
    for bm, models in sorted(score_table.items()):
        print(f"\n{bm}:")
        for m, s in sorted(models.items(), key=lambda x: -x[1]):
            print(f"  {m}: {s:.2f}%")

,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Benchmark,,,,,
click,68.30,73.88,94.96,66.82,75.90
haerae,63.20,69.87,77.01,57.14,16.00
kmmlu,52.24,57.51,86.48,53.37,62.50
kmmlu_hard,29.48,33.80,79.21,27.51,43.06
kobest_boolq,91.52,96.08,97.77,93.23,96.65


## Step 5: Category / Supercategory Aggregation

Korean benchmarks have hierarchical category structures:
- **KMMLU**: supercategory (STEM, HUMSS, Applied Science, Other) -> category (45 subjects)
- **CLIcK**: supercategory (Culture, Language) -> category (11 topics)

Category-level scores come from `category_accuracy.*` and `supercategory_accuracy.*` metrics.

In [32]:
def extract_category_scores(all_results: dict, benchmark_id: str, prefix: str = "category_accuracy.") -> dict:
    """Extract category-level scores for a specific benchmark.
    Returns {category: {model: score}}.
    """
    table = {}
    for model_name, jobs in all_results.items():
        for job_file, data in jobs.items():
            for bm in data.get("benchmarks", []):
                if bm.get("id") != benchmark_id:
                    continue
                metrics = bm.get("metrics", {})
                for key, value in metrics.items():
                    if key.startswith(prefix) and isinstance(value, (int, float)):
                        category = key[len(prefix):]
                        if category not in table:
                            table[category] = {}
                        table[category][model_name] = round(float(value), 2)
    return table


for bm_id in sorted(score_table.keys()):
    cat_scores = extract_category_scores(all_results, bm_id)
    super_scores = extract_category_scores(all_results, bm_id, prefix="supercategory_accuracy.")

    if super_scores:
        print(f"\n{bm_id} - Supercategory:")
        try:
            df_s = pd.DataFrame(super_scores).T.sort_index()
            df_s.index.name = "Supercategory"
            display(df_s.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
        except Exception:
            for cat, models in sorted(super_scores.items()):
                print(f"  {cat}: {models}")

    if cat_scores:
        print(f"\n{bm_id} - Category:")
        try:
            df_c = pd.DataFrame(cat_scores).T.sort_index()
            df_c.index.name = "Category"
            display(df_c.style.format("{:.2f}").highlight_max(axis=1, color="lightgreen"))
        except Exception:
            for cat, models in sorted(cat_scores.items()):
                print(f"  {cat}: {models}")


click - Supercategory:


,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Supercategory,,,,,
Culture,69.43,73.88,93.89,65.72,74.78
Language,65.79,73.89,97.88,69.27,78.38



click - Category:


,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Category,,,,,
Economy,89.83,93.22,94.83,81.36,91.53
Functional,70.71,85.71,98.11,82.35,89.52
Geography,78.23,80.33,98.21,71.20,77.78
Grammar,43.29,51.93,97.20,44.74,56.90
History,44.64,49.29,85.62,40.71,48.57
Law,58.45,64.84,95.74,56.62,65.75
Politics,79.76,79.76,93.75,77.38,85.71
Pop Culture,82.93,87.80,100.00,78.05,87.80
Society,81.23,86.41,96.59,80.58,89.97



haerae - Category:


...

,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Supercategory,,,,,
Other,29.48,33.80,79.21,27.51,43.06



kmmlu_hard - Category:


,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Category,,,,,
accounting,36.96,54.35,91.43,21.74,54.35
agricultural_sciences,19.00,31.00,66.67,14.00,35.00
aviation_engineering_and_maintenance,35.00,31.00,83.67,28.00,52.00
biology,30.00,27.00,87.04,24.00,36.00
chemical_engineering,31.00,28.00,90.43,24.00,50.00
chemistry,31.00,47.00,95.29,39.00,64.00
civil_engineering,27.00,29.00,78.05,34.00,41.00
computer_science,32.00,39.00,83.95,38.00,51.00
construction,28.00,27.00,67.07,27.00,26.00



kobest_boolq - Category:


,exaone4-32b,gemma4-12b,glm-53-flash,qwen3-14b,qwen36-27b
Category,,,,,
unknown,91.52,96.08,97.77,93.23,96.65


## Step 6: Export to Markdown

In [33]:
def score_table_to_markdown(score_table: dict, title: str = "Evaluation Results") -> str:
    """Convert score table to markdown format."""
    if not score_table:
        return "No results available."

    all_models = sorted(set(
        model for tasks in score_table.values() for model in tasks.keys()
    ))

    lines = [f"## {title}", ""]
    header = "| Benchmark | " + " | ".join(all_models) + " |"
    sep = "|:---" + "|---:" * len(all_models) + "|"
    lines.extend([header, sep])

    for bm in sorted(score_table.keys()):
        row = f"| {bm} |"
        for model in all_models:
            score = score_table[bm].get(model, "-")
            row += f" {score:.2f}% |" if isinstance(score, float) else f" {score} |"
        lines.append(row)

    return "\n".join(lines)


md_output = score_table_to_markdown(score_table, "Korean LLM Benchmark Results")
print(md_output)

md_path = RESULTS_DIR / "RESULTS.md"
md_path.write_text(md_output)
print(f"\nSaved to {md_path}")

## Korean LLM Benchmark Results

| Benchmark | exaone4-32b | gemma4-12b | glm-53-flash | qwen3-14b | qwen36-27b |
|:---|---:|---:|---:|---:|---:|
| click | 68.30% | 73.88% | 94.96% | 66.82% | 75.90% |
| haerae | 63.20% | 69.87% | 77.01% | 57.14% | 16.00% |
| kmmlu | 52.24% | 57.51% | 86.48% | 53.37% | 62.50% |
| kmmlu_hard | 29.48% | 33.80% | 79.21% | 27.51% | 43.06% |
| kobest_boolq | 91.52% | 96.08% | 97.77% | 93.23% | 96.65% |

Saved to /Users/hyochoi/dev/rhoai-evalhub-lab/results/RESULTS.md


## Step 7: Generate HTML Report

Use `generate_report.py` to produce a standalone HTML report with Chart.js visualizations.

In [34]:
!python generate_report.py --results-dir ../results --output ../results/report.html

print("\nOpen results/report.html in a browser to view the interactive report.")

Loading results from: /Users/hyochoi/dev/rhoai-evalhub-lab/results
Report generated: ../results/report.html
  Models: 5
  Tasks: 5

Open results/report.html in a browser to view the interactive report.
